# 08 — Segmentation Analysis

**Objetivo:** Segmentar motoristas e clientes em grupos distintos para direcionar ações específicas.

**Análises:**
1. **Segmentação de Motoristas** (K-Means) — perfis de risco operacional
2. **Segmentação de Clientes** (K-Means) — perfis de comportamento de compra
3. **Matriz de Prioridade** — onde focar os recursos para máximo impacto

> Segmentar permite ações personalizadas, mais eficientes e com maior ROI do que tratamentos genéricos.

In [ ]:
import sys
sys.path.append('..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

master = pd.read_parquet('../data/processed/master.parquet')
sns.set_theme(style='whitegrid')
print(f'Dataset: {master.shape[0]:,} pedidos')

## 1. Segmentação de Motoristas

**Features usadas:**
- `missing_rate` — taxa de falha
- `total_deliveries` — volume de trabalho
- `avg_order_value` — ticket médio dos pedidos atendidos
- `region_diversity` — número de regiões atendidas

In [ ]:
driver_profile = (
    master.groupby('driver_id')
    .agg(
        total_deliveries=('order_id', 'count'),
        missing_rate=('has_missing', 'mean'),
        avg_order_value=('order_amount', 'mean'),
        region_diversity=('region', 'nunique'),
        avg_items=('items_delivered', 'mean'),
    )
    .reset_index()
)
driver_profile = driver_profile[driver_profile['total_deliveries'] >= 3]

feats_d = ['missing_rate', 'total_deliveries', 'avg_order_value', 'region_diversity', 'avg_items']
X_d = driver_profile[feats_d].fillna(0)

scaler_d = StandardScaler()
X_d_sc = scaler_d.fit_transform(X_d)

# Elbow method
inertias = []
K_range = range(2, 9)
for k in K_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(X_d_sc)
    inertias.append(km.inertia_)

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(K_range, inertias, marker='o', color='steelblue', linewidth=2)
ax.set_title('Elbow Method — Segmentação de Motoristas', fontsize=12, fontweight='bold')
ax.set_xlabel('Número de Clusters (k)')
ax.set_ylabel('Inertia')
plt.tight_layout()
plt.savefig('../reports/figures/21_driver_elbow.png', dpi=150)
plt.show()

In [ ]:
# Aplicar com k=3 (3 perfis: alto/médio/baixo risco)
km_d = KMeans(n_clusters=3, random_state=42, n_init=10)
driver_profile['cluster'] = km_d.fit_predict(X_d_sc)

cluster_summary = (
    driver_profile.groupby('cluster')
    .agg(
        n_drivers=('driver_id', 'count'),
        avg_missing_rate=('missing_rate', 'mean'),
        avg_deliveries=('total_deliveries', 'mean'),
        avg_ticket=('avg_order_value', 'mean'),
    )
    .round(3)
)

# Nomear clusters por taxa de falha
cluster_summary = cluster_summary.sort_values('avg_missing_rate')
cluster_summary['perfil'] = ['Baixo Risco', 'Risco Médio', 'Alto Risco']

display(cluster_summary)

# Mapear rótulo de volta
label_map = cluster_summary['perfil'].to_dict()
ordered_clusters = cluster_summary.index.tolist()
driver_profile['perfil'] = driver_profile['cluster'].map(label_map)

# Visualização — scatter com PCA
pca = PCA(n_components=2, random_state=42)
coords = pca.fit_transform(X_d_sc)
driver_profile['pca1'] = coords[:, 0]
driver_profile['pca2'] = coords[:, 1]

colors_map = {'Baixo Risco': '#27ae60', 'Risco Médio': '#f39c12', 'Alto Risco': '#c0392b'}

fig, ax = plt.subplots(figsize=(11, 7))
for perfil, color in colors_map.items():
    subset = driver_profile[driver_profile['perfil'] == perfil]
    ax.scatter(subset['pca1'], subset['pca2'], c=color, label=perfil, alpha=0.6, s=30)

ax.set_title('Segmentação de Motoristas (K-Means k=3 | PCA 2D)',
             fontsize=13, fontweight='bold')
ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}% var)')
ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}% var)')
ax.legend(title='Perfil de Risco', fontsize=11)
plt.tight_layout()
plt.savefig('../reports/figures/22_driver_clusters.png', dpi=150)
plt.show()

print(f"\nDistribuicao de motoristas por perfil:")
for perfil, grp in driver_profile.groupby('perfil'):
    pct = len(grp) / len(driver_profile) * 100
    print(f"  {perfil}: {len(grp)} motoristas ({pct:.1f}%) | Taxa media: {grp['missing_rate'].mean()*100:.1f}%")

## 2. Segmentação de Clientes

**Features usadas:**
- `total_orders` — frequência de compra
- `total_spent` — valor total gasto
- `avg_order_value` — ticket médio
- `missing_rate` — frequência de problemas recebidos

In [ ]:
customer_profile = (
    master.groupby('customer_id')
    .agg(
        total_orders=('order_id', 'count'),
        total_spent=('order_amount', 'sum'),
        avg_order_value=('order_amount', 'mean'),
        missing_rate=('has_missing', 'mean'),
        customer_age=('customer_age', 'first'),
    )
    .reset_index()
)

feats_c = ['total_orders', 'total_spent', 'avg_order_value', 'missing_rate', 'customer_age']
X_c = customer_profile[feats_c].fillna(0)

scaler_c = StandardScaler()
X_c_sc = scaler_c.fit_transform(X_c)

km_c = KMeans(n_clusters=4, random_state=42, n_init=10)
customer_profile['cluster'] = km_c.fit_predict(X_c_sc)

cust_summary = (
    customer_profile.groupby('cluster')
    .agg(
        n_customers=('customer_id', 'count'),
        avg_orders=('total_orders', 'mean'),
        avg_spent=('total_spent', 'mean'),
        avg_ticket=('avg_order_value', 'mean'),
        avg_missing=('missing_rate', 'mean'),
        avg_age=('customer_age', 'mean'),
    )
    .round(2)
    .sort_values('avg_spent', ascending=False)
)

# Nomear clusters
cust_summary['segmento'] = ['VIP', 'Regular Alto', 'Regular Baixo', 'Ocasional']
display(cust_summary)

label_map_c = cust_summary['segmento'].to_dict()
customer_profile['segmento'] = customer_profile['cluster'].map(label_map_c)

# Visualização: bubble chart — frequência x ticket médio, tamanho = total gasto
fig, ax = plt.subplots(figsize=(11, 7))
seg_colors = {'VIP': '#8e44ad', 'Regular Alto': '#2980b9', 'Regular Baixo': '#27ae60', 'Ocasional': '#e67e22'}

for seg, color in seg_colors.items():
    s = customer_profile[customer_profile['segmento'] == seg]
    ax.scatter(s['total_orders'], s['avg_order_value'],
               s=s['total_spent'] / 5, c=color, alpha=0.5, label=seg)

ax.set_title('Segmentação de Clientes\n(x=frequência, y=ticket médio, tamanho=gasto total)',
             fontsize=13, fontweight='bold')
ax.set_xlabel('Total de Pedidos')
ax.set_ylabel('Ticket Médio ($)')
ax.legend(title='Segmento', fontsize=11)
plt.tight_layout()
plt.savefig('../reports/figures/23_customer_segments.png', dpi=150)
plt.show()

## 3. Matriz de Prioridade — Impacto Financeiro por Segmento de Motorista

In [ ]:
# Estimar impacto financeiro: cada falha = custo de reentrega + perda de item (~$25 estimado)
CUSTO_POR_FALHA = 25  # USD estimado por entrega com problema

# Juntar perfil de motorista de volta ao master
perf_map = driver_profile.set_index('driver_id')['perfil'].to_dict()
master['driver_perfil'] = master['driver_id'].map(perf_map)

impact = (
    master.groupby('driver_perfil')
    .agg(
        total_orders=('order_id', 'count'),
        total_failures=('has_missing', 'sum'),
        failure_rate=('has_missing', 'mean'),
        total_revenue=('order_amount', 'sum'),
    )
    .reset_index()
)
impact['estimated_cost'] = impact['total_failures'] * CUSTO_POR_FALHA
impact['pct_revenue_at_risk'] = impact['estimated_cost'] / impact['total_revenue'] * 100

print('IMPACTO FINANCEIRO ESTIMADO POR SEGMENTO DE MOTORISTA')
print(f'(Custo estimado por falha: ${CUSTO_POR_FALHA})')
print()
display(impact[['driver_perfil', 'total_orders', 'total_failures',
                'failure_rate', 'estimated_cost', 'pct_revenue_at_risk']]
        .assign(
            failure_rate=lambda d: (d['failure_rate']*100).round(1).astype(str)+'%',
            estimated_cost=lambda d: d['estimated_cost'].apply(lambda x: f'${x:,.0f}'),
            pct_revenue_at_risk=lambda d: d['pct_revenue_at_risk'].round(2).astype(str)+'%'
        ))

total_cost = impact['estimated_cost'].sum()
alto_risco_cost = impact[impact['driver_perfil'] == 'Alto Risco']['estimated_cost'].sum()
print(f"\nCusto total estimado com falhas: ${total_cost:,.0f}")
print(f"Custo atribuível a motoristas de Alto Risco: ${alto_risco_cost:,.0f} ({alto_risco_cost/total_cost*100:.1f}% do total)")

# Gráfico de impacto
order_seg = ['Alto Risco', 'Risco Médio', 'Baixo Risco']
impact_ordered = impact.set_index('driver_perfil').reindex(order_seg).reset_index()

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
colors_seg = ['#c0392b', '#f39c12', '#27ae60']

axes[0].bar(impact_ordered['driver_perfil'], impact_ordered['failure_rate']*100, color=colors_seg)
axes[0].set_title('Taxa de Falha por Perfil de Motorista', fontsize=12, fontweight='bold')
axes[0].set_ylabel('Taxa de Falha (%)')

axes[1].bar(impact_ordered['driver_perfil'], impact_ordered['estimated_cost'], color=colors_seg)
axes[1].set_title('Custo Estimado de Falhas por Perfil\n(USD)', fontsize=12, fontweight='bold')
axes[1].set_ylabel('Custo Estimado ($)')
axes[1].yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'${x:,.0f}'))

plt.tight_layout()
plt.savefig('../reports/figures/24_financial_impact_by_segment.png', dpi=150)
plt.show()